# Green screen removal on a GPU (open source)

This notebook runs the open-source [Zapmatte green screen remover](https://github.com/wsmhj/zapmatte-green-screen-remover) on a clip shot in front of a green screen and shows the result: a transparent WebM, a ProRes 4444 MOV, a black/white alpha matte and an MP4 on a new background colour.

How it works: [Robust Video Matting](https://github.com/PeterL1n/RobustVideoMatting) gives a soft alpha for the person in each frame; it is combined with a local chroma key, despill and tracking-marker removal.

Turn on **GPU** and **Internet** in the notebook settings. Prefer not to run code? The hosted version is [zapmatte.com](https://zapmatte.com) (first 10 seconds free).

In [ ]:
!git clone -q https://github.com/wsmhj/zapmatte-green-screen-remover
%cd zapmatte-green-screen-remover
!which ffmpeg || (apt-get -qq update && apt-get -qq install -y ffmpeg)
import torch; print('CUDA:', torch.cuda.is_available())

## A demo clip

A short green screen clip from Pexels (free to use).

In [ ]:
!wget -q -O clip.mp4 https://zapmatte.com/demo/suit-original.mp4
!ffprobe -v error -show_entries format=duration:stream=width,height -of default=nw=1 clip.mp4

## Remove the green screen

`--mov` writes ProRes 4444 with alpha (Premiere, After Effects, Final Cut, Resolve), `--matte` a black/white matte, `--color` an MP4 on a solid colour. Without flags you get a transparent WebM.

In [ ]:
%%time
!python remove_green_screen.py clip.mp4 --webm --mov --matte --color "#2255ff" --out-dir out
!ls -lh out

## Look at a frame

In [ ]:
import subprocess, matplotlib.pyplot as plt, matplotlib.image as mpimg
for src, name in [('clip.mp4', 'original'), ('out/clip-bg.mp4', 'new background'), ('out/clip-matte.mp4', 'alpha matte')]:
    subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-ss', '2', '-i', src, '-frames:v', '1', f'{name}.png'], check=True)
fig, ax = plt.subplots(1, 3, figsize=(12, 7))
for a, name in zip(ax, ['original', 'new background', 'alpha matte']):
    a.imshow(mpimg.imread(f'{name}.png'), cmap='gray'); a.set_title(name); a.axis('off')
plt.show()

## Watch the result

In [ ]:
from base64 import b64encode
from IPython.display import HTML
data = b64encode(open('out/clip-bg.mp4', 'rb').read()).decode()
HTML(f'<video width=360 controls loop autoplay muted src="data:video/mp4;base64,{data}"></video>')

## Use it on your own clips

Upload a video, then run `!python remove_green_screen.py your_clip.mp4 --mov`. Clips up to 4K work; on a T4 it runs about 19 fps at 1080p.

Code and license (GPL-3.0): https://github.com/wsmhj/zapmatte-green-screen-remover · Hosted version with no setup: https://zapmatte.com